# Parse the full corpus into data/parsed/

## Setup

Enabling autoreload and importing the parsing stack, declaring the raw/parsed/reference paths and the module name, and printing the GPU and the input file count to check the environment before a long run

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import re
import json
import time
import base64
from pathlib import Path

import torch
import pypdfium2 as pdfium
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field

PROJECT_ROOT  = Path(os.getcwd()).parent
PDF_PATH      = PROJECT_ROOT / "data" / "raw" / "pdfs"    
IPYNB_PATH    = PROJECT_ROOT / "data" / "raw" / "ipynb"
PARSED_DIR    = PROJECT_ROOT / "data" / "parsed"
REFERENCE_DIR = PROJECT_ROOT / "data" / "reference_slides"

MODUL = "machine_learning"

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("PDFs found:     ", len(list(PDF_PATH.glob("*.pdf"))))
print("Notebooks found:", len(list(IPYNB_PATH.glob("*.ipynb"))))

## Schema

Defining the Pydantic schema for a parsed slide (VlmSlideOutput) and for the final stored chunk (SlideChunk), so every parsed item is validated against one fixed structure

In [ ]:
class VlmSlideOutput(BaseModel):
    title: str = Field(default="")
    page_content: str = Field(default="")

class SlideChunk(BaseModel):
    id: str
    page_numbers: list[int]
    page_reference_path: str
    modul: str
    lecture: str
    title: str
    page_content: str


## VLM setup (slides)

Configuring the vision language model client and a function that sends a base64 encoded slide image with the system prompt and retries on failure, it returns validated JSON plus token usage

In [ ]:
from system_prompt import get_system_prompt

load_dotenv(PROJECT_ROOT / ".env", override=True)

client = OpenAI(
    base_url=os.getenv("GATEWAY_URL", ""),
    api_key=os.getenv("BEARER_TOKEN", ""),
)
model = os.getenv("VL_MODEL_GATEWAY", "")
assert model, "VL_MODEL_GATEWAY env-var missing"

SYSTEM_PROMPT = get_system_prompt()
MAX_RETRIES = 5
print("VLM-Model:", model)

def encode_image(image_path: Path) -> str:
    return base64.b64encode(image_path.read_bytes()).decode("utf-8")

def parse_slide_by_vlm(image_path: Path) -> tuple[VlmSlideOutput, int]:
    b64 = encode_image(image_path)
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "text", "text": "Beschreibe die folgende Vorlesungsfolie wie im Systemprompt gefordert"},
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    ]},
                ],
                response_format={"type": "json_object"},
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
                max_tokens=8192,
                temperature=0,
            )
            content = response.choices[0].message.content
            if not content:
                raise ValueError("empty content")
            total_tokens = response.usage.total_tokens if response.usage else 0
            return VlmSlideOutput.model_validate_json(content), total_tokens
        except Exception as e:
            print(f"[Retry {attempt + 1}/{MAX_RETRIES}] {image_path.name}: {e}")
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(1)

## Parse the slides

Rendering each PDF page to an image, parsing it with the VLM and persisting the chunks (skipping lectures that are already parsed), then running it over every PDF in the corpus

In [ ]:
def parse_presentation(pdf_path: Path) -> dict | None:
    lecture = pdf_path.stem
    out_reference = REFERENCE_DIR / MODUL / lecture
    out_json = PARSED_DIR / MODUL / lecture / f"{lecture}_chunks.json"

    if out_json.exists():
        print(f"skip   | {lecture} (exists: {out_json.name})")
        return None

    out_reference.mkdir(parents=True, exist_ok=True)
    out_json.parent.mkdir(parents=True, exist_ok=True)

    chunks: list[SlideChunk] = []
    tokens_total = 0
    t_start = time.perf_counter()
    with pdfium.PdfDocument(pdf_path) as pdf:
        total = len(pdf)
        for i, page in enumerate(pdf):
            img_path = out_reference / f"page_{i + 1}.png"
            page.render(scale=2).to_pil().save(img_path)

            print(f"  {lecture} [{i + 1:>3}/{total}] -> VLM ...", end=" ", flush=True)
            vlm_data, tokens = parse_slide_by_vlm(img_path)
            tokens_total += tokens
            print("...ok")

            chunks.append(SlideChunk(
                id=f"{lecture}_page_{i + 1}",
                page_numbers=[i + 1],
                page_reference_path=img_path.relative_to(PROJECT_ROOT).as_posix(),
                modul=MODUL,
                lecture=lecture,
                **vlm_data.model_dump(),
            ))

    elapsed = time.perf_counter() - t_start
    out_json.write_text(
        json.dumps([c.model_dump() for c in chunks], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"done   | {lecture}: {len(chunks)} chunks")
    return {"Vorlesung": lecture, "Folien": len(chunks),
            "Zeit (s)": round(elapsed, 1), "Tokens": tokens_total}

pdfs = sorted(PDF_PATH.glob("*.pdf"))
print(f"{len(pdfs)} PDFs:\n")

metrics = []
for pdf in pdfs:
    result = parse_presentation(pdf)
    if result is not None:
        metrics.append(result)

print("\nSlides Done!")

Aggregating per lecture timing and token counts into a DataFrame with a total row and writing parse_metrics.csv, to document the cost of the VLM parsing step

In [ ]:
import pandas as pd

EVAL_OUT = PROJECT_ROOT / "data" / "eval"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

if metrics:
    df = pd.DataFrame(metrics)
    df["Ø s/Folie"]   = (df["Zeit (s)"] / df["Folien"]).round(2)
    df["Ø Tok/Folie"] = (df["Tokens"]   / df["Folien"]).round().astype(int)

    gesamt = {
        "Vorlesung":   "Σ Gesamt",
        "Folien":      df["Folien"].sum(),
        "Zeit (s)":    round(df["Zeit (s)"].sum(), 1),
        "Tokens":      df["Tokens"].sum(),
        "Ø s/Folie":   round(df["Zeit (s)"].sum() / df["Folien"].sum(), 2),
        "Ø Tok/Folie": int(round(df["Tokens"].sum() / df["Folien"].sum())),
    }
    df = pd.concat([df, pd.DataFrame([gesamt])], ignore_index=True)

    df.to_csv(EVAL_OUT / "parse_full_lecture.csv", index=False, encoding="utf-8")
    print("gespeichert:", EVAL_OUT / "parse_full_lecture.csv")
    display(df)
else:
    print("no new lectures parsed (all skipped). No metrics.")


## Parse the notebooks

Defining parse_notebook to turn a Jupyter notebook into section level chunks (markdown headings as boundaries, code fenced), then parsing every source notebook, the second corpus modality next to the slides

In [ ]:
def _cell_source(cell: dict) -> str:
    src = cell.get("source", "")
    return "".join(src) if isinstance(src, list) else src

def _notebook_language(nb: dict) -> str:
    return nb.get("metadata", {}).get("language_info", {}).get("name", "python")

def parse_notebook(path: Path) -> list[SlideChunk]:
    nb = json.loads(path.read_text(encoding="utf-8"))
    language = _notebook_language(nb)
    lecture = path.stem

    notebook_title = None
    sections: list[tuple[str | None, list[str]]] = []
    section_title, parts = None, []
    for cell in nb.get("cells", []):
        text = _cell_source(cell).strip()
        if not text:
            continue
        if cell.get("cell_type") == "code":
            parts.append(f"```{language}\n{text}\n```")
            continue
        first_line = text.splitlines()[0].strip()
        if notebook_title is None and first_line.startswith("#"):
            notebook_title = first_line.lstrip("#").strip()
        if first_line.startswith("## "):
            sections.append((section_title, parts))
            section_title = first_line.lstrip("#").strip()
            parts = [text]
        else:
            parts.append(text)
    sections.append((section_title, parts))
    sections = [(t, p) for t, p in sections if p]

    if len(sections) >= 2:
        first_title, first_parts = sections[0]
        second_title, second_parts = sections[1]
        merged_first = (first_title, first_parts + second_parts)
        sections = [merged_first] + sections[2:]

    notebook_title = notebook_title or lecture
    chunks: list[SlideChunk] = []
    for i, (section_title, parts) in enumerate(sections, 1):
        if section_title and section_title != notebook_title:
            title = f"{notebook_title} — {section_title}"
        else:
            title = notebook_title
        chunks.append(SlideChunk(
            id=f"{lecture}_section_{i}",
            page_numbers=[i],
            page_reference_path="",
            modul=MODUL,
            lecture=lecture,
            title=title,
            page_content="\n\n".join(parts),
        ))
    return chunks

out_nb_dir = PARSED_DIR / "notebooks"
out_nb_dir.mkdir(parents=True, exist_ok=True)

ipynbs = sorted(IPYNB_PATH.glob("*.ipynb"))
print(f"{len(ipynbs)} Notebooks:\n")
for nb_path in ipynbs:
    chunks = parse_notebook(nb_path)
    out_path = out_nb_dir / f"{nb_path.stem}.json"
    out_path.write_text(
        json.dumps([c.model_dump() for c in chunks], indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print(f"done   | {nb_path.stem}: {len(chunks)} Sektionen")
print("\nNotebooks fertig.")

## Overview

Scanning all parsed JSON files for a quick corpus overview: chunks per file, slides vs. notebooks, and how many text paragraphs, images ([GRAFIK]), formulas ([FORMEL]) and code blocks ([CODE]) the parsed slides contain

In [ ]:
import re
from collections import Counter

BLOCK_SPLIT = re.compile(r"\n\s*\n")
MARKERS = {"[GRAFIK]": "images", "[FORMEL]": "formulas", "[CODE]": "code"}

def count_blocks(page_content: str) -> Counter:
    counts = Counter()
    for block in BLOCK_SPLIT.split(page_content):
        block = block.strip()
        if not block:
            continue
        prefix = next((m for m in MARKERS if block.startswith(m)), None)
        counts[MARKERS.get(prefix, "text")] += 1
    return counts

all_json       = sorted(PARSED_DIR.rglob("*.json"))
slide_files    = [f for f in all_json if (PARSED_DIR / MODUL) in f.parents]
notebook_files = [f for f in all_json if f not in slide_files]

slide_chunks, content = 0, Counter()
for f in slide_files:
    for c in json.loads(f.read_text(encoding="utf-8")):
        slide_chunks += 1
        content += count_blocks(c["page_content"])

notebook_sections = sum(len(json.loads(f.read_text(encoding="utf-8"))) for f in notebook_files)

for jf in all_json:
    n = len(json.loads(jf.read_text(encoding="utf-8")))
    print(f"{n:>3} chunks | {jf.relative_to(PARSED_DIR)}")

print(f"\nSlides:    {len(slide_files):>3} lectures  -> {slide_chunks} slides")
print(f"Notebooks: {len(notebook_files):>3} notebooks -> {notebook_sections} sections")

print("\nSlide content elements:")
print(f" Text paragraphs: {content['text']:>4}")
print(f" Images [GRAFIK]: {content['images']:>4}")
print(f" Formulas [FORMEL]: {content['formulas']:>4}")
print(f" Code [CODE]: {content['code']:>4}")

total_chunks = slide_chunks + notebook_sections
print(f"\nOverall: {len(all_json)} files, {total_chunks} chunks in {PARSED_DIR}")

## Post processing

Post processing every parsed file: stripping page number footers, dropping near empty chunks and writing the cleaned corpus to parsed_clean/, with a report of what was stripped and dropped

In [ ]:
PARSED_CLEAN_DIR = PROJECT_ROOT / "data" / "parsed_clean"
PAGE_NUM_LINE    = re.compile(r"^\s*(Seite|Folie|Page)\s+\d+\s*$", re.I)  

def strip_footer(text: str) -> tuple[str, list[str]]:
    kept, removed = [], []
    for line in text.splitlines():
        s = line.strip()
        if PAGE_NUM_LINE.match(s):
            removed.append(s)
            continue
        kept.append(line)
    return "\n".join(kept).strip(), removed

def is_empty(content: str) -> bool:
    return len(content.strip()) < 30 and "[GRAFIK]" not in content

def relative_reference(path: str) -> str:
    if path and Path(path).is_absolute():
        return Path(path).relative_to(PROJECT_ROOT).as_posix()
    return path

def clean_file(chunks: list[dict]) -> tuple[list[dict], list[dict], list[str]]:
    kept, dropped, stripped = [], [], []
    for c in chunks:
        c = dict(c)
        c["page_content"], removed = strip_footer(c.get("page_content", ""))
        c["page_reference_path"] = relative_reference(c.get("page_reference_path", ""))
        stripped += removed
        if is_empty(c["page_content"]):
            dropped.append({**c, "drop_reason": "empty"})
        else:
            kept.append(c)
    return kept, dropped, stripped

PARSED_CLEAN_DIR.mkdir(parents=True, exist_ok=True)
report, dropped_all, stripped_all = [], [], []
for jf in sorted(PARSED_DIR.rglob("*.json")):
    raw = json.loads(jf.read_text(encoding="utf-8"))
    kept, dropped, stripped = clean_file(raw)
    out = PARSED_CLEAN_DIR / jf.relative_to(PARSED_DIR)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(kept, indent=2, ensure_ascii=False), encoding="utf-8")
    report.append({"file": str(jf.relative_to(PARSED_DIR)),
                   "raw": len(raw), "kept": len(kept), "dropped": len(dropped),
                   "page_nums_stripped": len(stripped)})
    dropped_all.extend(dropped)
    stripped_all.extend(stripped)

print(f"── page-number lines stripped: {len(stripped_all)} total ──")
for row in report:
    if row["page_nums_stripped"]:
        print(f"  {row['page_nums_stripped']:>3}  {row['file']}")

print("\n── dropped chunks (empty only) ──")
for d in dropped_all:
    print(f"  {d['lecture']} p.{d.get('page_numbers')} — {d['title']}")